# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FawadAhmad-bilal/flyrank-assignment-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #10 (ML Appendix, p.27) — "What Predicts Health?" (Random Forest feature importance).** The paper's own Health Score is defined as Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts). The Random Forest then reports Average Position (43%), Impressions (32%), Scroll Depth (15%), and CTR (8%) as the top predictors of that same Health Score. **Where does the label come from?** Directly from a weighted sum of those four inputs. **Does the validation design carry the claim?** No — the paper itself flags this ("the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal"), but an 80/20 holdout split cannot rescue label-derived features (leakage type 1 in the hunting-leakage skill). A held-out test set still contains the same circular relationship on every row. This isn't a data-snooping error a different split would catch — it's built into the feature choice itself.

**Finding #11 (ML Appendix, p.29) — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy).** Growth/decline is labeled from 30-day-vs-previous-30-day impression change — the exact same label shape as my own `is_declining_label`. `Impressions` (recent) appears as a model FEATURE in the same chart. **Where does the label come from?** The change in impressions across two 30-day windows. **Does the validation design carry the claim?** Only partly. An 80/20 holdout guards against overfitting *across rows*, but if 'Impressions' as a feature is measured from a window that overlaps the label's own 30-day comparison window, that's leakage type 2 (future/overlapping windows) — and a random holdout split doesn't fix within-row window overlap, because every row, train or test, has the same overlap. The paper doesn't specify whether the feature window is fully separated from the label window, which is exactly the kind of ambiguity the hunting-leakage skill says to draw out as a timeline before trusting the number.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running my Week-5 model (Logistic Regression + Random Forest, same features) under two splits on the same data: a naive random row split (BEFORE — ignores that rows repeat within client_id), and the grouped-by-client split I actually used in Week 5 (AFTER).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df["word_count"] = df.groupby("content_type")["word_count"].transform(lambda s: s.fillna(s.median()))
df["main_intent"] = df["main_intent"].fillna("unknown")

numeric_features = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
                     "engagement_rate", "ai_traffic_pct", "content_age_days",
                     "days_since_last_update", "word_count"]
categorical_features = ["content_type", "main_intent"]
X = df[numeric_features + categorical_features]
y = df["is_declining_label"]
groups = df["client_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def fit_and_eval(X_train, X_test, y_train, y_test):
    prep_lr = ColumnTransformer([("num", StandardScaler(), numeric_features), ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])
    lr = Pipeline([("prep", prep_lr), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))]).fit(X_train, y_train)
    prep_rf = ColumnTransformer([("num", "passthrough", numeric_features), ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])
    rf = Pipeline([("prep", prep_rf), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1))]).fit(X_train, y_train)
    lr_probs, rf_probs = lr.predict_proba(X_test)[:,1], rf.predict_proba(X_test)[:,1]
    return pd.DataFrame([{"k": k, "base_rate": round(y_test.mean(),3),
                           "LR": round(precision_at_k(lr_probs, y_test.values, k),3),
                           "RF": round(precision_at_k(rf_probs, y_test.values, k),3)} for k in [20,50,100,500]])

# BEFORE: naive random row split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y)
before = fit_and_eval(Xtr, Xte, ytr, yte)
print("BEFORE -- naive random split (ignores repeated client_id rows):")
print(before.to_string(index=False))

# AFTER: grouped split by client_id (same as Week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups))
after = fit_and_eval(X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx])
print("\nAFTER -- grouped split by client_id:")
print(after.to_string(index=False))

gap = (before[["LR","RF"]] - after[["LR","RF"]])
gap.insert(0, "k", before["k"])
print("\nGAP (random minus grouped) -- the memorization this reveals:")
print(gap.to_string(index=False))

BEFORE -- naive random split (ignores repeated client_id rows):
  k  base_rate    LR    RF
 20      0.542 0.550 0.900
 50      0.542 0.600 0.900
100      0.542 0.640 0.910
500      0.542 0.694 0.852



AFTER -- grouped split by client_id:
  k  base_rate   LR    RF
 20      0.517 0.80 0.550
 50      0.517 0.66 0.600
100      0.517 0.58 0.620
500      0.517 0.53 0.626

GAP (random minus grouped) -- the memorization this reveals:
  k     LR    RF
 20 -0.250 0.350
 50 -0.060 0.300
100  0.060 0.290
500  0.164 0.226


**The gap is the finding.** Random Forest at precision@20 reads 0.900 under a random split — but drops to 0.550 under the honest client-grouped split, a 0.350 collapse. That gap is RF memorizing client-specific patterns it had already seen rows from during training; it was never really achieving 90% precision on genuinely unseen clients. Logistic Regression is far more stable across both splits (and even reads *higher* at k=20 under the grouped split, 0.550 → 0.800 — likely noise from the smaller effective test set at that k, not a real ability to generalize better than random would suggest). **Conclusion: the grouped-split numbers from Week 5 are the ones worth trusting; the random-split RF number would have been a materially overstated claim if reported instead.**

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Re-running the hunting-leakage-and-validating checklist against my FINAL feature set (`impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `engagement_rate`, `ai_traffic_pct`, `content_age_days`, `days_since_last_update`, `word_count`, `content_type`, `main_intent`):

- [x] **Timeline drawn:** every feature is a 90-day rolling aggregate knowable at prediction time; the label (`trend_direction`, 30d-vs-prev-30d) sits in a separate, later comparison window.
- [x] **No label-derived or sibling columns in features:** `trend_direction` / `trend_pct` are used only to build `is_declining_label`, checked explicitly below by the train-with/train-without test.
- [x] **No product flags as features:** FlyRank's own `Optimization Flags` (Healthy / Fix CTR / Fix Content / Zombie Page) are not in the raw file at all, so this can't accidentally happen here.
- [ ] → [x] **Population selection checked:** I use all 30,000 rows regardless of activity level; the low-visibility rows are handled by score, not dropped, so there's no "active only" survivorship filter hiding dead pages.
- [x] **Split grouped by the repeating entity:** `client_id`, confirmed no client overlap in Week 5 and again in Section 2 above.
- [x] **Base rate printed next to every metric:** done in every precision@K table so far.
- [x] **Top feature importance sanity-checked:** Week 5 found `impressions_90d` at 26%, not a 90%+ single-feature spike — checked again below by directly testing removal of the two suspect columns.
- [x] **Metrics recomputed out-of-fold:** all precision@K numbers come from the held-out test split, never from training rows.

**The train-with/train-without test** (per the skill: "train once WITH the suspect, once WITHOUT — a collapse from ~1.0 to ~0.7 is the confession"):

In [2]:
# Deliberately ADD the suspect leaky column and see if the score jumps toward 1.0 --
# if it doesn't, the harness itself is broken (this is the skill's own verification method).
df["trend_pct"] = df["trend_pct"].fillna(0)
leaky_numeric = numeric_features + ["trend_pct"]  # trend_pct is literally used to build the label

prep_leak = ColumnTransformer([("num", "passthrough", leaky_numeric),
                                 ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])
X_leaky = df[leaky_numeric + categorical_features]
rf_leaky = Pipeline([("prep", prep_leak),
                      ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1))])
rf_leaky.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_probs = rf_leaky.predict_proba(X_leaky.iloc[test_idx])[:, 1]

print("WITH the leaky column (trend_pct) added back in:")
for k in [20, 50, 100, 500]:
    print(f"precision@{k}: {precision_at_k(leaky_probs, y.iloc[test_idx].values, k):.3f}")

imp = pd.Series(rf_leaky.named_steps["clf"].feature_importances_,
                index=leaky_numeric + list(rf_leaky.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(categorical_features)))
print("\nTop feature:", imp.sort_values(ascending=False).index[0], f"({imp.max():.1%} importance)")

WITH the leaky column (trend_pct) added back in:
precision@20: 1.000
precision@50: 1.000
precision@100: 1.000
precision@500: 1.000

Top feature: trend_pct (87.7% importance)


**Confession confirmed:** precision@K jumps from ~0.55-0.80 (clean) straight to a perfect 1.000 at every K once `trend_pct` is added back in, with `trend_pct` alone carrying 87.7% of the Random Forest's feature importance. That's exactly the pattern the skill describes as a confession, not a win — it confirms both that `trend_pct` is genuinely leaky (as expected, since it's the label's own source column) and that this test harness correctly detects leakage when it's actually present, which is what makes the earlier clean AFTER numbers trustworthy rather than an artifact of a harness that can't tell the difference.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest sentence from my Week-5 notebook (Section 3):** *"Random Forest starts weaker at k=20 (0.550) but pulls ahead as the queue gets longer... suggesting it's picking up more nuanced, non-linear patterns further down the list."*

**Problem with this sentence:** "suggesting it's picking up more nuanced, non-linear patterns" claims a mechanism (WHY the model wins) that a precision@K table alone can't support — that's an inference about model behavior dressed as an observation. It also doesn't carry the Section 2 finding above, which shows RF's advantage is partly inflated by a memorization gap when measured under an imperfect split.

**Rewritten in safe language (observed / measured / directional / decision-support):** "Under the client-grouped test split, Random Forest is **directionally** stronger than Logistic Regression at larger queue depths (**observed** precision@100 = 0.620 vs. 0.580, precision@500 = 0.626 vs. 0.530), while Logistic Regression **measured** better at the very top of the queue (precision@20 = 0.800 vs. 0.550). Neither model's edge is large enough, on this single client-held-out split, to claim one architecture is definitively better — the practical, **decision-support** reading is: use Logistic Regression for short weekly review lists, Random Forest if the review queue routinely runs deeper than ~100 pages."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.